# Lección 10.4 — Arquitecturas icónicas: LeNet, AlexNet, VGG, ResNet

> **Módulo 10 · Visión por computadora (CNN)** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 70 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 10.3 (tu primera CNN), Lección 9.7 (gradientes que desaparecen), Lección 9.8 (batch normalization, dropout)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Contar la historia de las CNN a través de cuatro redes: **LeNet, AlexNet, VGG y ResNet**, y qué idea aportó cada una.
- Explicar por qué **dos filtros de 3 × 3** son mejor negocio que uno de 5 × 5 (la idea de VGG).
- Explicar el problema de las redes muy profundas y cómo lo resuelven las **conexiones residuales** (los "atajos" de ResNet).
- Comparar arquitecturas por parámetros, velocidad y exactitud, y elegir una según el caso.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> 🖥️ Todo corre en CPU en pocos minutos (también en Google Colab sin GPU). Si tienes GPU, se usará sola.

## ▶️ Preparación

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, models

plt.rcParams["figure.figsize"] = (8, 4.5)
mnist = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)
X_tr = mnist.data[:10000].float().div(255).unsqueeze(1); y_tr = mnist.targets[:10000]
X_te = mnist_test.data[:2000].float().div(255).unsqueeze(1); y_te = mnist_test.targets[:2000]

contar = lambda m: sum(p.numel() for p in m.parameters())

def entrenar(modelo, epocas=2, X=X_tr, y=y_tr, opt=None):
    """Entrena en lotes de 64 y devuelve la exactitud en prueba al final de cada época."""
    torch.manual_seed(0)
    opt = opt or torch.optim.Adam(modelo.parameters(), lr=1e-3)
    historia = []
    for _ in range(epocas):
        modelo.train(); orden = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            idx = orden[i:i + 64]
            perdida = nn.functional.cross_entropy(modelo(X[idx]), y[idx])
            opt.zero_grad(); perdida.backward(); opt.step()
        modelo.eval()
        with torch.no_grad():
            historia.append((modelo(X_te).argmax(1) == y_te).float().mean().item())
    return historia

print("✅ Todo listo")

---
## 1. 📖 La intuición: la evolución de los aviones

Los hermanos Wright volaron en 1903 con un avión de madera y tela. Los aviones de hoy usan los **mismos principios** (alas, sustentación, motor), pero cada generación aportó **una idea clave**: el metal, el motor a reacción, los materiales compuestos…

Con las CNN pasa igual. Todas usan las piezas de las lecciones 10.2 y 10.3 (convolución, ReLU, pooling, capa densa), pero cada arquitectura famosa aportó **una idea** que permitió ir más lejos:

```
1998  LeNet-5   ─ "Convolución + pooling funciona"            (leer códigos postales y cheques)
2012  AlexNet   ─ "Más grande + GPU + ReLU + dropout"          (gana ImageNet por goleada)
2014  VGG       ─ "Solo filtros 3×3, pero muchísimas capas"     (simple y profunda)
2015  ResNet    ─ "Atajos para poder entrenar 100+ capas"      (supera al humano en ImageNet)
```

**ImageNet** es el gran examen: 1.2 millones de fotos de **1,000** categorías (perros de 120 razas, autos, frutas…). Su competencia anual (ILSVRC) se mide con el **error top-5**: el porcentaje de fotos en las que la respuesta correcta **no** está entre las 5 apuestas del modelo.

| Año | Ganador | Error top-5 |
|---|---|---|
| 2011 | Métodos sin redes neuronales | ≈25.8 % |
| 2012 | **AlexNet** (8 capas) | ≈15.3 % |
| 2014 | **VGG** (19 capas, 2.º lugar) · GoogLeNet (22 capas, 1.º) | ≈7.3 % · ≈6.7 % |
| 2015 | **ResNet** (152 capas) | ≈3.6 % |
| — | Un humano entrenado (estimación de A. Karpathy) | ≈5.1 % |

### ✋ Ejercicio 1 — Lee la tabla
1. ¿Cuánto bajó el error de 2011 a 2012? ¿Por qué crees que 2012 se considera el "año cero" del deep learning?
2. ¿Qué tendencia ves entre el número de capas y el error?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. De ≈25.8 % a ≈15.3 %: más de **10 puntos** de golpe, cuando los años anteriores se mejoraba de a 1–2 puntos. Fue tan contundente que, en un par de años, casi toda la comunidad de visión por computadora se pasó a las redes neuronales.
2. **Más capas, menos error**: 8 → 19/22 → 152. Pero, como verás en la Sección 4, apilar capas **no** era tan fácil: hizo falta un truco (los atajos de ResNet) para pasar de ≈20 a más de 100.
</details>

---
## 2. 📖 LeNet-5 (1998): la abuela de todas

Yann LeCun la diseñó para leer **códigos postales y cheques**, y llegó a procesar una parte importante de los cheques de EE. UU. Tiene exactamente la estructura que construiste en la Lección 10.3:

```
Dígito 28×28 ─► Conv 6@5×5 ─► Pool ─► Conv 16@5×5 ─► Pool ─► Densa 120 ─► Densa 84 ─► 10 dígitos
```

Las diferencias con una CNN moderna son detalles de su época: usaba **tanh** en lugar de ReLU y **average pooling** en lugar de max pooling.

In [ ]:
torch.manual_seed(0)
lenet = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Tanh(), nn.AvgPool2d(2),   # 28 → 28 → 14
    nn.Conv2d(6, 16, kernel_size=5), nn.Tanh(), nn.AvgPool2d(2),             # 14 → 10 → 5
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.Tanh(),
    nn.Linear(120, 84), nn.Tanh(),
    nn.Linear(84, 10),
)
for capa in lenet:
    if contar(capa) > 0:
        print(f"{str(capa):<55} {contar(capa):>7,} parámetros")
print(f"{'TOTAL':<55} {contar(lenet):>7,}")

### 🔮 Ejercicio 2 — ¿Dónde están los parámetros?
Mirando la tabla de arriba: ¿qué porcentaje aproximado de los parámetros está en las capas **convolucionales** y cuál en las **densas**? ¿Te sorprende?

✍️ *Mi estimación:*

Después entrena LeNet 2 épocas (▶️ entre 10 y 30 s):

In [ ]:
inicio = time.time()
historia = entrenar(lenet)
print(f"Exactitud en prueba por época: {[f'{h:.1%}' for h in historia]} · {time.time() - inicio:.0f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

Las convolucionales tienen 156 + 2,416 = 2,572 parámetros (≈**4 %**), y las densas ≈59,000 (≈**96 %**). Es típico de las CNN clásicas: **la mayoría de los parámetros está en la parte densa**, aunque la mayoría del **cómputo** esté en las convoluciones. Esta observación será importante para AlexNet y VGG… y por eso las redes modernas cambian la parte densa por *pooling* global.

LeNet llega a ≈90 % con solo 2 épocas y 10,000 dígitos (con tanh y average pooling aprende algo más lento que la CNN de la Lección 10.3).
</details>

---
## 3. 📖 AlexNet (2012) y VGG (2014): más grandes, más profundas

**AlexNet** (Krizhevsky, Sutskever y Hinton) era "una LeNet gigante" con cuatro ingredientes que hoy son estándar:
- **ReLU** en lugar de tanh: entrena varias veces más rápido (Lección 9.3).
- **Dropout** en las capas densas, contra el sobreajuste (Lección 9.8).
- **Data augmentation**: recortes y reflejos aleatorios de las fotos (Lección 9.8).
- **GPU**: la entrenaron durante ~una semana con dos tarjetas gráficas de videojuegos.

**VGG** (Simonyan y Zisserman, Oxford) se preguntó: *¿y si usamos **solo** filtros de 3 × 3, pero muchísimas capas?* Su diseño es tan regular que se describe en una línea: bloques de 2–3 convoluciones 3 × 3 + max pooling, duplicando los filtros (64 → 128 → 256 → 512 → 512).

### 📖 ¿Por qué solo 3 × 3?
Dos convoluciones de 3 × 3 seguidas tienen el mismo **campo receptivo** que una de 5 × 5 (Lección 10.3: 3 → 5). Pero:
- Usan **menos parámetros**.
- Tienen **dos** ReLU en lugar de una: más capacidad de combinar patrones.

### 🔮 Ejercicio 3 — Haz las cuentas
Con 64 canales de entrada y 64 de salida (sin sesgos), ¿cuántos parámetros tiene…
1. una capa de 5 × 5?
2. dos capas de 3 × 3?
3. Y para el mismo campo receptivo que un 7 × 7: ¿una de 7 × 7 o **tres** de 3 × 3?

✍️ *Mis cuentas:*

In [ ]:
C = 64   # ✏️ prueba con otros números de canales
una_5x5 = nn.Conv2d(C, C, 5, bias=False)
dos_3x3 = nn.Sequential(nn.Conv2d(C, C, 3, bias=False), nn.Conv2d(C, C, 3, bias=False))
una_7x7 = nn.Conv2d(C, C, 7, bias=False)
tres_3x3 = nn.Sequential(*[nn.Conv2d(C, C, 3, bias=False) for _ in range(3)])

for nombre, capa in [("1 × (5×5)", una_5x5), ("2 × (3×3)", dos_3x3), ("1 × (7×7)", una_7x7), ("3 × (3×3)", tres_3x3)]:
    print(f"{nombre}: {contar(capa):>8,} parámetros")

<details><summary><b>👀 Ver respuesta</b></summary>

1. 64 × 64 × 25 = **102,400**.
2. 2 × (64 × 64 × 9) = **73,728**: un **28 % menos**.
3. Una de 7 × 7: 64 × 64 × 49 = **200,704**; tres de 3 × 3: 3 × 36,864 = **110,592**, casi la **mitad**.

Misma "vista" de la imagen, menos parámetros y más no linealidades. Por eso, desde VGG, casi todas las CNN usan filtros de 3 × 3.
</details>

### 📖 El precio del tamaño
`torchvision` trae estas arquitecturas listas. Creémoslas **sin pesos entrenados** (`weights=None`), solo para medir su tamaño y su velocidad. ▶️ VGG16 tarda unos segundos en construirse: ¡tiene que crear 138 millones de números al azar!

### 🔮 Ejercicio 4 — ¿Quién es más pesada? ¿Y más lenta?
Ordena estas redes de **menos a más parámetros**: AlexNet (8 capas), VGG16 (16 capas), ResNet18 (18 capas), ResNet50 (50 capas). ¿La más profunda es la que más parámetros tiene?

✍️ *Mi orden:*

In [ ]:
fotos = torch.rand(4, 3, 224, 224)          # 4 fotos de 224×224, el tamaño estándar de ImageNet
for nombre in ["alexnet", "vgg16", "resnet18", "resnet50"]:
    red = getattr(models, nombre)(weights=None).eval()
    with torch.no_grad():
        red(fotos)                                       # una pasada de "calentamiento"
        inicio = time.time(); red(fotos); seg = time.time() - inicio
    print(f"{nombre:<9} {contar(red) / 1e6:6.1f} millones de parámetros · {seg * 1000 / 4:5.0f} ms por foto (CPU)")
    del red

<details><summary><b>👀 Ver respuesta</b></summary>

De menos a más parámetros: **ResNet18 (≈11.7 M) → ResNet50 (≈25.6 M) → AlexNet (≈61.1 M) → VGG16 (≈138.4 M)**.

¡La más profunda (ResNet50) tiene **5 veces menos** parámetros que VGG16! ¿Por qué? AlexNet y VGG terminan con capas densas enormes (la primera densa de VGG16 tiene ≈103 millones de parámetros, ¡el 74 % del total!). ResNet usa *pooling* global (Lección 10.3) y una sola capa densa pequeña.

En velocidad, VGG16 es de lejos la más lenta (sus convoluciones trabajan con muchos filtros sobre mapas grandes); ResNet18 es varias veces más rápida que VGG16. AlexNet también es rápida a pesar de sus 61 M de parámetros: tiene pocas convoluciones, y multiplicar por una capa densa grande es barato comparado con convolucionar mapas grandes. **Parámetros ≠ velocidad.** Los milisegundos exactos dependen de tu computadora (y de qué más esté haciendo).
</details>

---
## 4. 📖 ¿Qué pasa si apilamos muchas más capas? El muro de la profundidad

Si más capas = menos error, ¿por qué no hacer una VGG de 100 capas? Porque **no entrena**. En la Lección 9.7 viste que, al retropropagar el error capa por capa, la señal se va multiplicando por números pequeños y **se desvanece**: las primeras capas casi no reciben gradiente y no aprenden.

*Batch normalization* (Lección 9.8) ayudó, pero aun así los investigadores de Microsoft observaron algo raro en 2015: una red "plana" de **56** capas tenía **más error** que una de 20, **incluso en entrenamiento**. No era sobreajuste: la red profunda simplemente no lograba optimizarse (lo llamaron el problema de **degradación**).

### 📖 La idea de ResNet: atajos
Kaiming He y su equipo propusieron un cambio mínimo. En vez de que cada bloque calcule la salida completa:

```
Bloque normal:     x ──► [capas] ──► F(x)                         salida = F(x)

Bloque residual:   x ──► [capas] ──► F(x) ──(+)──►                salida = x + F(x)
                   │                         ▲
                   └────── atajo ────────────┘
```

**Analogía:** un corrector de estilo no reescribe todo tu texto: te lo devuelve **igual** y solo marca **los cambios**. Cada bloque residual hace eso: copia su entrada y le **suma una corrección** `F(x)`. Si un bloque no tiene nada útil que aportar, le basta con aprender `F(x) ≈ 0` y deja pasar la información intacta.

Y el atajo es también una **autopista para el gradiente**: el error puede viajar hacia atrás directamente por los atajos, sin desvanecerse.

### 🔮 Ejercicio 5 — Plana vs. residual
Para verlo en segundos usamos redes **densas** muy profundas con activación tanh (la que más sufre el desvanecimiento, Lección 9.7). Entrenaremos con 3, 20 y 50 capas ocultas, **con** y **sin** atajos, y mediremos:
- el **gradiente** que llega a la primera capa al empezar;
- la **exactitud** en prueba tras 3 épocas.

Antes de ejecutar: ¿qué crees que le pasa a la red **plana de 50 capas**? ¿Y a la residual de 50?

✍️ *Mis predicciones:*

In [ ]:
class Bloque(nn.Module):
    def __init__(self, ancho, con_atajo):
        super().__init__()
        self.capa = nn.Sequential(nn.Linear(ancho, ancho), nn.Tanh())
        self.con_atajo = con_atajo
    def forward(self, x):
        return x + self.capa(x) if self.con_atajo else self.capa(x)    # ← toda la diferencia está aquí

def red_profunda(n_capas, con_atajo):
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 32), nn.ReLU(),
                         *[Bloque(32, con_atajo) for _ in range(n_capas)], nn.Linear(32, 10))

X_5k, y_5k = X_tr[:5000], y_tr[:5000]
for n_capas in [3, 20, 50]:                         # ✏️ prueba con 100
    for con_atajo in [False, True]:
        red = red_profunda(n_capas, con_atajo)
        nn.functional.cross_entropy(red(X_5k[:256]), y_5k[:256]).backward()
        grad = red[1].weight.grad.norm().item()      # tamaño del gradiente en la PRIMERA capa
        red.zero_grad()
        hist = entrenar(red, epocas=3, X=X_5k, y=y_5k, opt=torch.optim.SGD(red.parameters(), lr=0.05))
        print(f"{n_capas:>2} capas {'CON atajos' if con_atajo else 'planas    '} · gradiente en la 1.ª capa: {grad:8.1e} · "
              f"exactitud tras 3 épocas: {hist[-1]:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Con **3 capas**, las dos aprenden (la residual bastante más rápido: ≈87 % contra ≈54 %).
- Las redes **planas de 20 y 50 capas no aprenden nada**: se quedan en ≈11–12 %, lo mismo que adivinar al azar entre 10 dígitos. Mira el gradiente: ≈0.000004 con 20 capas y ≈0.0000000000001 (10⁻¹³) con 50. La primera capa **no recibe ninguna señal** de qué corregir.
- Las **residuales** de 20 y 50 capas sí aprenden (≈86 % y ≈78 %): su gradiente en la primera capa es de ≈6 y ≈13, porque viaja por la "autopista" de los atajos.

Un cambio de **una línea** (`x + ...`) separa una red que no entrena de una que sí. Esa línea permitió entrenar redes de 152 capas en 2015, y hoy está en casi todas las redes profundas, incluidos los Transformers (Lección 12.3).
</details>

### 📖 El bloque residual de verdad
En ResNet, la "corrección" `F(x)` son dos convoluciones 3 × 3 con *batch normalization*:

```
x ─► Conv 3×3 ─► BatchNorm ─► ReLU ─► Conv 3×3 ─► BatchNorm ─(+)─► ReLU ─► salida
│                                                            ▲
└───────────────────────── atajo ────────────────────────────┘
```

In [ ]:
class BloqueResidual(nn.Module):
    def __init__(self, canales):
        super().__init__()
        self.F = nn.Sequential(
            nn.Conv2d(canales, canales, 3, padding=1, bias=False), nn.BatchNorm2d(canales), nn.ReLU(),
            nn.Conv2d(canales, canales, 3, padding=1, bias=False), nn.BatchNorm2d(canales))
    def forward(self, x):
        return torch.relu(x + self.F(x))          # entrada + corrección

bloque = BloqueResidual(16)
x = torch.rand(1, 16, 14, 14)
print("Entrada:", tuple(x.shape), "→ salida:", tuple(bloque(x).shape))
print("Parámetros del bloque:", f"{contar(bloque):,}")

### ✋ Ejercicio 6 — La condición del atajo
Para poder hacer `x + F(x)`, ¿qué tiene que cumplir `F(x)` respecto a `x`? ¿Por qué todas las convoluciones del bloque usan `padding=1`?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

`F(x)` debe tener **exactamente la misma forma** que `x` (mismos canales, mismo alto y ancho): solo se pueden sumar tensores del mismo tamaño, número a número. Por eso las convoluciones usan `padding=1` con kernel 3 (conservan el tamaño, Lección 10.2) y no cambian el número de canales.

¿Y cuando ResNet necesita reducir el tamaño o duplicar los canales? Entonces el atajo lleva una pequeña convolución de **1 × 1** con *stride* 2 que "ajusta" `x` a la nueva forma. Lo verás en la práctica integradora.
</details>

### 📖 Por dentro de ResNet18
Así está organizada la ResNet18 de `torchvision`, la que usarás preentrenada en la Lección 10.5:

In [ ]:
resnet18 = models.resnet18(weights=None)
x = torch.zeros(1, 3, 224, 224)
for nombre, parte in resnet18.named_children():
    if nombre == "fc":
        x = torch.flatten(x, 1)
    x = parte(x)
    print(f"{nombre:<8} → salida {str(tuple(x.shape)):<20} {contar(parte):>11,} parámetros")

### ✋ Ejercicio 7 — Lee la arquitectura
1. ¿Qué le pasa al alto y ancho de los mapas de `layer1` a `layer4`? ¿Y al número de canales? ¿Qué regla de la Lección 10.3 reconoces?
2. ¿Qué hace `avgpool`? ¿Por qué la capa final `fc` es tan pequeña comparada con la de VGG?
3. ¿En qué parte están la mayoría de los parámetros?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El tamaño se divide entre 2 en cada etapa (56 → 28 → 14 → 7) y los canales se **duplican** (64 → 128 → 256 → 512): es la regla "tras reducir, duplica los filtros".
2. `avgpool` es un **pooling global**: convierte los 512 mapas de 7 × 7 en **512 números**. Por eso `fc` solo necesita 512 × 1,000 + 1,000 ≈ **513,000** parámetros, frente a los ≈103 millones de la primera densa de VGG16.
3. En `layer4` (≈8.4 millones, el 72 % del total): son las convoluciones con 512 filtros. Esas capas profundas guardan los patrones más complejos (partes de objetos).
</details>

---
## 5. 📖 Las perillas de una arquitectura

Al elegir o diseñar una CNN, estas son las grandes decisiones:

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Profundidad** (nº de capas/bloques) | Cuántos niveles de abstracción y qué tan grande es el campo receptivo | Más capacidad para objetos complejos; más lenta; sin atajos, no entrena | Más rápida; puede quedarse corta (subajuste) | ResNet18–50 para fotos; 2–4 bloques para imágenes pequeñas |
| **Ancho** (filtros por capa) | Cuántos patrones distintos por nivel | Más capacidad, mucho más cómputo y memoria | Más ligera | 32–64 en la primera etapa y duplicar |
| **Tamaño del kernel** | Vista de cada filtro | Más parámetros por capa | Más barato; apila capas para ver más | 3 × 3 (desde VGG) |
| **Resolución de entrada** | Detalle que la red puede ver | Detecta objetos pequeños; cómputo ∝ píxeles | Más rápida, pierde detalle | 224 × 224 en ImageNet; 32 × 32 en CIFAR |
| **Cabeza** (parte final) | Cómo se pasa de mapas a la decisión | Densas grandes = muchos parámetros (AlexNet/VGG) | Pooling global + 1 densa | Pooling global + 1 densa (ResNet) |

### 🔮 Ejercicio 8 — La resolución cuesta
El costo de una convolución crece con el **número de píxeles**. Si pasas de fotos de 224 × 224 a 448 × 448, ¿cuántas veces más lenta será, más o menos, ResNet18? ¿Y cuántos parámetros más tendrá?

In [ ]:
resnet18.eval()
for lado in [112, 224, 448]:                       # ✏️
    fotos = torch.rand(2, 3, lado, lado)
    with torch.no_grad():
        resnet18(fotos); inicio = time.time(); resnet18(fotos); seg = time.time() - inicio
    print(f"{lado}×{lado}: {seg * 1000 / 2:6.0f} ms por foto · parámetros: {contar(resnet18):,}")

<details><summary><b>👀 Ver respuesta</b></summary>

Al duplicar el lado, los píxeles se multiplican por **4**, así que en teoría el tiempo crece **≈4 veces**. En la práctica verás un aumento algo menor (entre 2 y 4 veces), porque parte del tiempo es fijo y la medición en CPU es ruidosa, pero la tendencia es clara. Los parámetros **no cambian** (11,689,512): los filtros son los mismos, y gracias al pooling global la capa final tampoco depende del tamaño. Más resolución = más detalle, pero se paga en tiempo y memoria.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Red muy profunda que no aprende (se queda en el azar) | Gradientes que se desvanecen; red "plana" | Atajos residuales + *batch normalization* (o usa una ResNet) |
| Red profunda **peor** que una menos profunda, incluso en entrenamiento | Problema de degradación | Conexiones residuales |
| Modelo enorme y lento para una app de celular | Elegir VGG/ResNet50 "porque son famosas" | Arquitecturas ligeras (MobileNet, EfficientNet) |
| *The size of tensor a must match the size of tensor b* en `x + F(x)` | El bloque cambia canales o tamaño | Atajo con convolución 1 × 1 (y *stride*) |
| Sobreajuste brutal con pocas imágenes | Arquitectura de millones de parámetros entrenada desde cero | *Transfer learning* (Lección 10.5) |

### 🔮 Ejercicio 9 — Más grande no siempre es mejor
Una startup quiere clasificar fotos de platillos **en el celular** del usuario, sin internet. Tiene 3,000 fotos etiquetadas. Duda entre VGG16 y ResNet18. Con lo que mediste en esta lección, ¿cuál le recomiendas y por qué? ¿Hay algo mejor que entrenarla desde cero?

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

**ResNet18** sin duda frente a VGG16: ≈12 veces menos parámetros (≈47 MB contra ≈550 MB en memoria) y varias veces más rápida, con mejor exactitud en ImageNet. Para un celular incluso convendría una arquitectura diseñada para eso (**MobileNet**, ≈2.5 millones de parámetros).

Y con solo 3,000 fotos, entrenarla **desde cero** sobreajustaría: lo mejor es partir de una red **preentrenada** en ImageNet y ajustarla. Es el *transfer learning* de la próxima lección.
</details>

---
## 7. 📋 Ficha comparativa de arquitecturas

| | LeNet-5 | AlexNet | VGG16 | ResNet18 / 50 |
|---|---|---|---|---|
| **Año** | 1998 | 2012 | 2014 | 2015 |
| **Capas con pesos** | 5 | 8 | 16 | 18 / 50 (hasta 152) |
| **Parámetros** | ≈62 mil | ≈61 M | ≈138 M | ≈11.7 M / ≈25.6 M |
| **Idea clave** | Conv + pooling + densa | ReLU, dropout, augmentation, GPU | Solo 3 × 3, muy profunda y regular | **Atajos residuales**, BatchNorm, pooling global |
| **✅ Fortaleza** | Simple y didáctica | Demostró el poder del deep learning | Diseño simple, buenas características | Profunda, precisa y eficiente; estándar de la industria |
| **❌ Debilidad** | Solo para imágenes pequeñas | Superada; capas densas enormes | Pesadísima y lenta | Hay arquitecturas más eficientes para celulares |
| **Úsala hoy para…** | Aprender | Historia | Aprender; extracción de características | **Punto de partida por defecto** en *transfer learning* |

## 8. 🧠 Resumen en 5 puntos
1. Todas las CNN famosas combinan las mismas piezas (conv, ReLU, pooling, densa); cada una aportó **una idea**.
2. **LeNet** (1998) demostró la receta; **AlexNet** (2012) la escaló con GPU, ReLU, dropout y augmentation, y cambió la historia.
3. **VGG**: dos filtros 3 × 3 ven lo mismo que uno 5 × 5 con menos parámetros → redes de solo 3 × 3, muy profundas.
4. Las redes planas muy profundas **no entrenan** (el gradiente se desvanece). **ResNet** suma un **atajo**: salida = x + F(x), una autopista para la información y el gradiente.
5. Profundidad ≠ parámetros: ResNet50 es 5 veces más pequeña que VGG16 gracias al **pooling global**. Elige la arquitectura por precisión, **velocidad y tamaño**, no por fama.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué NO fue una novedad importante de AlexNet?
- a) Usar ReLU · b) Entrenar en GPU · c) Usar conexiones residuales · d) Usar dropout

**2.** Tres convoluciones 3 × 3 seguidas tienen el mismo campo receptivo que…
- a) una de 3 × 3 · b) una de 5 × 5 · c) una de 7 × 7 · d) una de 9 × 9

**3.** En un bloque residual, la salida es…
- a) F(x) · b) x · F(x) · c) x + F(x) · d) F(F(x))

**4.** ¿Por qué VGG16 tiene muchos más parámetros que ResNet50, aunque sea menos profunda?
- a) Usa kernels de 7 × 7
- b) Termina con capas densas gigantes en lugar de pooling global
- c) Usa más canales de entrada
- d) Tiene conexiones residuales

**5.** Una red plana de 50 capas se queda en 11 % de exactitud en MNIST y el gradiente de su primera capa es ≈10⁻¹³. El diagnóstico más probable es…
- a) Sobreajuste · b) Tasa de aprendizaje demasiado baja · c) Gradientes que se desvanecen · d) Datos mal etiquetados

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Los atajos residuales llegaron con ResNet, en 2015.
2. **c)** 3 → 5 → 7.
3. **c)** La entrada más la "corrección".
4. **b)** La primera densa de VGG16 tiene ≈103 millones de parámetros.
5. **c)** El 11 % es azar puro y el gradiente es prácticamente cero: la señal no llega a las primeras capas. Solución: atajos (y/o BatchNorm, ReLU, buena inicialización, Lección 9.7).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Con un bloque residual, si las capas de `F` aprenden a dar siempre 0, el bloque deja pasar `x` sin cambios. ¿Por qué eso garantiza (en teoría) que una ResNet de 50 capas **no** debería ser peor que una de 20?

**B. Caso 💼** Un hospital tiene 200,000 radiografías etiquetadas y servidores con GPU; necesita la máxima exactitud y el tiempo de respuesta no importa mucho. Una app de agricultura necesita reconocer plagas en hojas desde celulares baratos en el campo, sin internet. Recomienda una familia de arquitecturas para cada uno y justifica en dos frases.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque la red de 50 capas siempre **puede imitar** a la de 20: le basta con que sus 30 bloques extra aprendan `F(x) = 0` (dejar pasar). Y aprender "0" es fácil (pesos pequeños). En una red plana, en cambio, imitar a la de 20 exigiría que las 30 capas extra aprendan a **copiar exactamente** su entrada, algo muy difícil de lograr con capas que multiplican y aplican ReLU. Así que, con atajos, más profundidad "en el peor caso no estorba".

**B.** **Hospital:** una red profunda y precisa (ResNet50 o mayor, o arquitecturas más recientes; incluso Vision Transformers, Lección 10.6), idealmente preentrenada y ajustada con sus 200,000 imágenes: tiene datos y cómputo de sobra, y lo que importa es el error. **Agricultura:** una red **ligera** diseñada para celulares (MobileNet o EfficientNet pequeña), preentrenada y ajustada con *transfer learning*: lo crítico es el tamaño, la batería y la velocidad sin conexión, a costa de unos puntos de exactitud.
</details>

### 9.3 🧩 Práctica integradora: "Construye tu mini-ResNet"

Vas a construir una ResNet en miniatura para MNIST y compararla con LeNet.

**Paso 1 🔮 — Cuando la forma cambia.** Queremos un bloque que **duplique los canales** (16 → 32) y **reduzca el tamaño a la mitad** (14 → 7). Antes de ejecutar: ¿por qué fallará `x + F(x)` si no hacemos nada con el atajo?

In [ ]:
F_cambia = nn.Sequential(nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32))
x = torch.rand(1, 16, 14, 14)
print("x    :", tuple(x.shape))
print("F(x) :", tuple(F_cambia(x).shape))
try:
    x + F_cambia(x)
except RuntimeError as e:
    print("❌", str(e)[:100])

<details><summary><b>👀 Ver respuesta</b></summary>

`x` es `(1, 16, 14, 14)` y `F(x)` es `(1, 32, 7, 7)`: no tienen la misma forma, así que no se pueden sumar número a número.
</details>

**Paso 2 ✏️ — Arregla el atajo.** La solución de ResNet: que el atajo también pase por una convolución **1 × 1** con *stride* 2 que convierta `x` a la forma de `F(x)`. Completa `kernel_size` y `stride` del atajo.

In [ ]:
atajo = nn.Conv2d(16, 32, kernel_size=1, stride=1)     # ✏️ ¿qué stride necesitas?
try:
    print("atajo(x) + F(x):", tuple((atajo(x) + F_cambia(x)).shape))
except RuntimeError as e:
    print("❌ Todavía no:", str(e)[:80])

<details><summary><b>👀 Ver respuesta</b></summary>

Con `stride=2` el atajo da `(1, 32, 7, 7)` y la suma funciona. La convolución 1 × 1 no mira vecinos: solo "mezcla" los 16 canales para producir 32 y, con *stride* 2, se salta una posición de cada dos. Es el atajo "de proyección" que ResNet usa al principio de `layer2`, `layer3` y `layer4` (en `torchvision` se llama `downsample`).
</details>

**Paso 3 🔮 — La mini-ResNet.** Ahora montamos una red pequeña: una convolución inicial, dos bloques residuales (los de la Sección 4), pooling global y una sola capa densa.
1. ¿Tendrá más o menos parámetros que LeNet (61,706)?
2. ¿Crees que será mejor o peor que LeNet tras 2 épocas?

▶️ Tarda entre 20 s y 1 min.

In [ ]:
torch.manual_seed(0)
mini_resnet = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 28 → 14
    BloqueResidual(16), nn.MaxPool2d(2),                           # 14 → 7
    BloqueResidual(16),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),                         # pooling global → 16 números
    nn.Linear(16, 10))
print(f"Parámetros → mini-ResNet: {contar(mini_resnet):,} · LeNet: {contar(lenet):,}")
inicio = time.time()
hist = entrenar(mini_resnet)
print(f"Exactitud por época → mini-ResNet: {[f'{h:.1%}' for h in hist]} · LeNet: {[f'{h:.1%}' for h in historia]} · {time.time() - inicio:.0f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Muchos menos**: ≈9,700, unas **6 veces menos** que LeNet. Casi todos están en las convoluciones de los bloques; la capa densa final solo tiene 170.
2. Quedan **casi empatadas** (≈88 % la mini-ResNet contra ≈90 % LeNet tras 2 épocas). Con 6 veces menos parámetros, eso es un buen resultado: es la lección de ResNet (pooling global en vez de densas gigantes). Con más épocas o más filtros, la mini-ResNet mejora; en MNIST todas las CNN terminan por encima del 98 %.
</details>

**Paso 4 ✋ — Elige arquitectura.** Para cada caso, elige entre **LeNet**, **ResNet18 preentrenada**, **ResNet50 preentrenada** y **MobileNet preentrenada**, y justifica en una frase:
1. Leer dígitos de formularios escaneados en blanco y negro (28 × 28).
2. Clasificar 50 tipos de defectos en fotos de alta resolución de piezas industriales, con servidores con GPU.
3. Reconocer especies de aves con una app de celular sin conexión.

✍️ *Mis elecciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **LeNet** (o una CNN pequeña como la de la Lección 10.3): imágenes diminutas y simples; una red grande sería desperdicio.
2. **ResNet50 preentrenada** (o ResNet18 si hay pocas fotos): tarea difícil, fotos grandes y cómputo disponible; empezar desde ImageNet ahorra datos.
3. **MobileNet preentrenada**: lo crítico es el tamaño y la velocidad en el teléfono.

No hay una única respuesta: lo importante es justificar con **datos, cómputo y requisitos** (velocidad, tamaño, exactitud).
</details>

---
## 📚 Para profundizar
- 📘 **Artículo:** He et al. (2015), *Deep Residual Learning for Image Recognition*: el artículo de ResNet; la Figura 1 muestra la degradación (56 capas peor que 20).
- 📘 **Artículo:** Krizhevsky, Sutskever y Hinton (2012), *ImageNet Classification with Deep Convolutional Neural Networks* (AlexNet).
- 📺 **Video:** Andrej Karpathy, curso CS231n de Stanford, clase "CNN Architectures" (inglés, disponible en YouTube).
- 📘 **Documentación:** `torchvision.models`: todas las arquitecturas, con sus parámetros y su exactitud en ImageNet.

---
⬅️ **Anterior:** [10.3 Pooling y jerarquía de características](10.3-pooling-y-jerarquia.ipynb) · ➡️ **Siguiente:** [10.5 Transfer learning y fine-tuning: no empezar de cero](10.5-transfer-learning.ipynb)